# LAB 2 — Modelagem de Dados para Analytics

**Disciplina** Engenharia de Dados para IA

**Aula** 2 — Organização e Manuseio de Dados Relacionais

**Duração do Lab** ~1h30 (em aula)

**Entrega** Diagrama ER + Normalização até 3FN + Star Schema + DDL SQL + justificativas + KPI

**Scripts de apoio** `lodlog_modelo_dados_v3.sql` e `lodlog_lab3_seed_v3`

1. A tabela `lodlog_op.entrega` é entregue **propositalmente em 2FN** — com colunas redundantes de cliente, veículo, motorista e CD (dependências transitivas). Os alunos devem **normalizar para 3FN**.
2. KPI `kpi_cat_atraso` na entrega (operacional) e na fato (DW). Categoria derivada de `minutos_atraso`.
3. Mini-dimensão `lodlog_dw.dim_cat_atraso` (Kimball, SCD Tipo 1, 5 linhas). Surrogate key `sk_cat_atraso` replicada na fato.

## Objetivos de Aprendizagem

1. Validar e corrigir um modelo entidade-relacionamento para um cenário de negócio.
2. Identificar dependências transitivas em uma tabela em 2FN e **normalizar até a 3FN**.
3. Validar um Star Schema para analytics e ML.
4. Adicionar um atributo de KPI em uma dimensão do Star Schema (mini-dimensão).
5. Justificar o uso de surrogate keys vs natural keys.
6. Identificar quando desnormalizar para ganho de performance.

## Parte 0 — Contexto e Setup (10 min)

Na Aula 1, o grupo mapeou o ecossistema de dados da LODLog. Agora é hora de **estruturar** esses dados.

***Problema do dia:*** *A LODLog tem dados espalhados em ERP, WMS, TMS e sensores IoT. Cada sistema usa suas próprias chaves e formatos. Não há uma visão unificada do cliente, do veículo ou do produto. Relatórios demoram porque precisam fazer JOINs complexos entre sistemas.*

**Missão do LAB 2:** Receber o modelo com a tabela `entrega` em **2FN** (proposital), normalizar até **3FN**, compreender o **Star Schema** com a **mini-dimensão** `dim_cat_atraso` e implementar o DDL correspondente.

### Antes de começar

1. No Databricks Free Edition, abra **Workspace** → **Create** → **Git folder** e informe o repositório `https://github.com/Escola-de-Matematica-Aplicada/EDIA-Labs-DataBricks.git`.
2. Na pasta `Labs`, rode o script `lodlog_modelo_dados_v3.sql`. Ele cria os databases `lodlog_op` (operacional) e `lodlog_dw` (analítico) e suas tabelas.
3. Em seguida, abra o notebook `lodlog_lab3_seed_v3` e clique em **Run all** (com **Serverless** selecionado). Ele leva cerca de 2 minutos e popula as tabelas com ~50.000 entregas fictícias — necessárias para inspecionar os dados e para as consultas da Parte 4.
4. Inspecione as tabelas no menu **Catalog** ou execute a célula abaixo.

In [ ]:
%sql
SHOW TABLES IN lodlog_op;

In [ ]:
%sql
SHOW TABLES IN lodlog_dw;

## Parte 1 — Modelo Entidade-Relacionamento (25 min)

### Atividade 1.1: Identifique as Entidades

A partir do cenário LODLog e das tabelas geradas em `lodlog_op`, liste pelo menos **8 entidades** com seus atributos principais. Use como referência o script `lodlog_modelo_dados_v3.sql` (ou `DESCRIBE lodlog_op.<tabela>`).

### Atividade 1.2: Relacionamentos e Cardinalidade

Atenção: nem todo par abaixo tem relacionamento **direto** no modelo. Se o relacionamento for indireto, indique por qual entidade ele passa.

| Entidade A | Entidade B | Cardinalidade | Descrição do Relacionamento |
| --- | --- | --- | --- |
| VEICULO | MOTORISTA |  |  |
| PEDIDO | CLIENTE |  |  |
| PEDIDO | VEICULO |  |  |
| PRODUTO | CENTRO_DISTRIBUICAO |  |  |
| MOTORISTA | MANUTENCAO |  |  |

---

**— linha de base —** Daqui para baixo é desejável/opcional.

## Parte 2 — Normalização até 3FN (25 min)

**ATENÇÃO —** a tabela `lodlog_op.entrega` está **propositalmente em 2FN**. Ela tem **dependências transitivas** (dados redundantes de outras entidades). Sua missão é normalizar até a **3FN**.

### Atividade 2.1: Identifique as Anomalias e Dependências Transitivas (individual, 10 min)

Inspecione a tabela `lodlog_op.entrega` no Databricks (célula abaixo). Ela tem a PK `entrega_id` (simples), portanto está em 1FN e 2FN (sem dependências parciais), mas tem **4 grupos de dependências transitivas**:

```
ENTREGA(entrega_id, pedido_id, veiculo_id, motorista_id,
  -- Dados redundantes do CLIENTE (dependência transitiva 1)
  cliente_id, cliente_cnpj, cliente_razao_social, cliente_segmento,
  -- Dados redundantes do VEÍCULO (dependência transitiva 2)
  veiculo_placa, veiculo_modelo, veiculo_fabricante, veiculo_capacidade_kg,
  -- Dados redundantes do MOTORISTA (dependência transitiva 3)
  motorista_nome, motorista_cnh, motorista_categoria,
  -- Dados redundantes do CD (dependência transitiva 4)
  cd_origem_id, cd_origem_codigo, cd_origem_nome, cd_origem_uf,
  data_saida, data_entrega_prevista, data_entrega_real,
  distancia_km, custo_combustivel, custo_pedagio, valor_frete,
  multa_atraso, status_entrega,
  kpi_cat_atraso)
```

In [ ]:
%sql
SELECT * FROM lodlog_op.entrega LIMIT 10;

**Responda:**

1. **Quantas anomalias** essa tabela tem se um cliente trocar de razão social? (anomalia de ____________)
2. **Quantos registros** precisam ser atualizados se o veículo 42 mudar de placa? (dica: você pode responder com um `SELECT COUNT(*)` na célula abaixo)
3. Se uma entrega for cancelada, **o que acontece com os dados do cliente** nessa entrega?
4. Liste as **4 dependências funcionais transitivas** no formato X → Y → Z (ex: `entrega_id → cliente_id → cliente_razao_social`).

In [ ]:
%sql
-- Quantos registros de entrega carregam a placa do veículo 42?

### Atividade 2.2: Normalize até a 3FN (grupo, 15 min)

Decomponha a tabela ENTREGA (que está em 2FN) até a **Terceira Forma Normal (3FN)**.

**Sua entrega deve conter:**

1. Lista de **tabelas finais** após a normalização (mínimo: cliente, veiculo, motorista, centro_distribuicao, entrega).
2. Para cada tabela: PK e FKs.
3. O DDL da tabela ENTREGA em 3FN (você vai implementá-lo na Atividade 4.1).

## Parte 3 — Star Schema para Analytics (30 min)

### Atividade 3.1: Diferença ER vs Star Schema (individual, 5 min)

| Característica | Modelo ER (Transacional) | Star Schema (Analítico) |
| --- | --- | --- |
| Objetivo principal |  |  |
| Número de tabelas |  |  |
| Tipo de JOIN |  |  |
| Granularidade |  |  |
| Uso de surrogate keys |  |  |
| Otimizado para |  |  |

### Atividade 3.2: Compreenda o Star Schema da LODLog (grupo, 15 min)

A partir do ER normalizado (Parte 2), compreenda o **Star Schema** para responder às perguntas de negócio abaixo. Use o `lodlog_modelo_dados_v3.sql` como referência das tabelas DW existentes.

Para as perguntas de negócio abaixo, indique quais tabelas do DW poderiam trazer a resposta. Não precisa rodar consulta ou obter resultado.

1. Qual a taxa de pontualidade por centro de distribuição e por mês?
2. Qual o custo médio de frete por região e por tipo de veículo?
3. Quantos pedidos foram entregues com atraso por cliente no último trimestre?
4. Qual a distribuição de atrasos por categoria (Adiantado / No Prazo / Atraso Leve / Moderado / Crítico) por mês?
5. Qual a correlação entre a categoria de atraso e o score de segurança do motorista?

**Requisitos:** tabela fato bem definida, pelo menos **5 dimensões** (4 originais + a nova `dim_cat_atraso`), granularidade clara.

### Atividade 3.3: Surrogate Keys (individual, 5 min)

| Tabela | Usa surrogate key? | Por quê? |
| --- | --- | --- |
| Fato entregas |  |  |
| Dimensão Tempo |  |  |
| Dimensão Cliente |  |  |
| Dimensão Veículo |  |  |
| Dimensão Centro de Distribuição |  |  |
| Dimensão Categoria Atraso |  |  |

### Atividade 3.4: Mini-dimensão do KPI (grupo, 5 min)

O script de seed v3 já criou a mini-dimensão `lodlog_dw.dim_cat_atraso` com 5 registros. Responda:

1. **Por que** a categoria de atraso foi para uma **mini-dimensão** (e não para a `dim_tempo` ou `dim_motorista`)?
2. A coluna `kpi_cat_atraso` aparece **duas vezes** no modelo: na `entrega` (operacional) e na `fato_entregas` (DW). Isso é **redundância** aceitável? Justifique.
3. A coluna `sla_violado` na mini-dimensão é uma **flag booleana**. Ela poderia virar uma **dimensão separada** (SCD Tipo 1)?

> **Conceito-chave — Mini-dimensão (Kimball):** atributos de **baixa cardinalidade** que são **estáveis** (mudam raramente) e usados em filtros/agrupamentos de BI devem ser isolados em uma **mini-dimensão**, em vez de serem replicados como string na fato. Isso permite:
>
> * Performance: a fato é menor (INT em vez de STRING)
> * Flexibilidade: trocar a descrição da categoria sem alterar a fato
> * Conformidade: o valor da categoria passa a ser controlado pela dimensão

In [ ]:
%sql
SELECT * FROM lodlog_dw.dim_cat_atraso ORDER BY ordem;

## Parte 4 — SQL DDL (20 min)

### Atividade 4.1: Crie a Nova Tabela Operacional (individual, 10 min)

Você já normalizou a tabela `entrega` na Atividade 2.2. Agora **aplique a normalização em SQL**:

1. Consulte o `CREATE TABLE lodlog_op.entrega` (em 2FN) no script `lodlog_modelo_dados_v3.sql`.
2. Na célula abaixo, crie a sua versão normalizada em 3FN como `lodlog_op.entrega_3fn` — apenas com PK, FKs e medidas (sem colunas redundantes).
3. (Opcional) Popule a nova tabela com `INSERT INTO lodlog_op.entrega_3fn SELECT ... FROM lodlog_op.entrega` e compare as duas com `DESCRIBE`.

> Criamos uma tabela nova em vez de alterar a original para não quebrar o seed, que recarrega a `entrega` em 2FN.

In [ ]:
%sql
-- Escreva aqui o DDL da ENTREGA em 3FN
CREATE OR REPLACE TABLE lodlog_op.entrega_3fn (
  entrega_id BIGINT NOT NULL PRIMARY KEY
  -- complete com as FKs e as medidas
)
USING DELTA;

### Atividade 4.2: Valide a Mini-Dimensão no DW (individual, 5 min) (opcional)

A mini-dimensão `dim_cat_atraso` já foi criada. Mas vocês precisam garantir a **integridade**:

1. Faça uma query que conte quantas entregas há em cada categoria (JOIN de `fato_entregas` com `dim_cat_atraso`).

In [ ]:
%sql
-- Entregas por categoria de atraso

2. Faça uma query que mostre a taxa de pontualidade por região (JOIN de `fato_entregas` com `dim_centro_distribuicao`). Use como ponto de partida:

In [ ]:
%sql
SELECT c.regiao,
       SUM(CASE WHEN f.indicador_atraso = 0 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS taxa_pontualidade
-- complete com FROM, JOIN e GROUP BY

### Atividade 4.3: Justifique suas Escolhas (grupo, 5 min)

1. Por que escolheu essa granularidade para a tabela fato?
2. Qual dimensão vai crescer mais? Como lidaria com isso?
3. Se o negócio pedir uma nova categoria "Atraso Severíssimo" (> 24h), o que muda no modelo?

---

**Dica do Professor:** O Star Schema não é "melhor" que o ER — são modelos para propósitos diferentes. O ER serve para operar (transações); o Star Schema serve para analisar (agregações). Um pipeline de dados bem projetado começa no ER dos sistemas transacionais (em 3FN) e termina no Star Schema do DW — com mini-dimensões para KPIs de baixa cardinalidade.